## Model Setup with Quantization

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Model ID
model_id = "tiiuae/falcon-rw-1b"

# Quantization configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Load Model and Tokenizer

In [2]:
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [3]:
# Set padding token and side for causal LM
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Verify quantization
print(f"Model dtype: {model.dtype}")
if hasattr(model, "hf_quantizer"):
    print(f"Quantization method: {model.hf_quantizer.quantization_config.__class__.__name__}")
else:
    print("Model is not quantized")

Model dtype: torch.bfloat16
Quantization method: BitsAndBytesConfig


## Test Inference

In [4]:
# Test inference
prompt = "Explain quantum computing"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    # repetition_penalty=1.2
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Explain quantum computing
In a previous article we examined how the concept of quantum computing was introduced. In this article we will go into more detail about quantum computing.
The first article on quantum computing explained how quantum computing works.
In this article we will go into more detail about quantum computing.
What is quantum computing?
Quantum computing is a type of computing where quantum mechanics is applied to the calculation process.
Quantum computing is not a new idea. In fact, it has been around since the
